In [ ]:
!pip install -q gradio

In [ ]:
import os

os.environ.pop("GOOGLE_API_KEY", None)

print("cleared.")


cleared.


In [ ]:
from getpass import getpass

def setup_api_key():
    k = os.environ.get("GOOGLE_API_KEY", "")

    if not k or k.startswith("$"):
        k = getpass("Please enter your Google API Key: ").strip()
        os.environ["GOOGLE_API_KEY"] = k

    print("✅ Key captured.")
    print("length =", len(k))
    print("starts_with =", k[:4], "ends_with =", k[-4:])
    print("has_space =", any(c.isspace() for c in k))

setup_api_key()

Please enter your Google API Key: ··········
✅ Key captured.
length = 39
starts_with = AIza ends_with = -gwc
has_space = False


In [ ]:
!pip install -q langchain-google-genai google-generativeai gradio pandas Pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.9/515.9 kB 13.2 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import pandas as pd
import json
import os
import re
import io
import base64
from datetime import datetime, timedelta
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from PIL import Image


In [ ]:
# --- 1. SETUP ---
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"

DB_FILE = "pantry_memory.json"
USED_LOG = "used_items.json"

In [ ]:
# --- 2. DATA UTILS ---
def load_json(filename):
    if not os.path.exists(filename): return []
    with open(filename, "r") as f:
        try:
            data = json.load(f)
            return data if isinstance(data, list) else []
        except: return []

def save_json(data, filename):
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

def calculate_status(date_added, shelf_life):
    try:
        added = datetime.strptime(str(date_added), "%Y-%m-%d")
        expiry = added + timedelta(days=int(shelf_life))
        days_left = (expiry - datetime.now()).days
        if days_left < 0: return "🛑 Expired"
        if days_left < 3: return "⚠️ At Risk"
        return "✅ Fresh"
    except:
        return "❓ Unknown"

def get_inventory_df():
    data = load_json(DB_FILE)
    cols = ["Item", "Date Added", "Expiration Date", "Status"]
    if not data: return pd.DataFrame(columns=cols)

    df = pd.DataFrame(data)
    if "Expiration Date" not in df.columns:
        df["Expiration Date"] = df.apply(lambda x: (datetime.strptime(x["Date Added"], "%Y-%m-%d") + timedelta(days=int(x.get("Shelf_Life", 7)))).strftime("%Y-%m-%d"), axis=1)

    df['Status'] = df.apply(lambda x: calculate_status(x['Date Added'], x.get('Shelf_Life', 7)), axis=1)
    return df[cols]

In [ ]:
# --- 3. AGENT LOGIC ---
def inventory_agent(text, img):
    if not os.environ.get("GOOGLE_API_KEY"):
        return "❌ Missing API Key.", get_inventory_df()

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

    prompt = """
    You are a grocery receipt expert. Your task is to extract ONLY food and beverage items.

    1. Look for item names, ignoring prices, tax codes, and store metadata.
    2. Decipher shorthand abbreviations.
    3. Ignore non-food items.
    4. Estimate 'Shelf_Life' in days.

    Return ONLY a valid JSON array: [{"Item": "Name", "Shelf_Life": 7}]
    """

    try:
        content = [{"type": "text", "text": prompt}]
        if text: content.append({"type": "text", "text": f"Manual Text: {text}"})

        if img:
            # Convert PIL image to base64 so Gemini can read it
            buffered = io.BytesIO()
            img.save(buffered, format="JPEG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

            content.append({
                "type": "image_url",
                "image_url": f"data:image/jpeg;base64,{img_base64}"
            })


        response = llm.invoke([HumanMessage(content=content)])

        json_str = response.content.replace('```json', '').replace('```', '').strip()
        match = re.search(r"\[.*\]", json_str, re.DOTALL)

        if not match: return "❌ No items detected.", get_inventory_df()

        new_items = json.loads(match.group())
        db = load_json(DB_FILE)
        for item in new_items:
            item["Date Added"] = datetime.now().strftime("%Y-%m-%d")
            item["Expiration Date"] = (datetime.now() + timedelta(days=int(item.get("Shelf_Life", 7)))).strftime("%Y-%m-%d")
            db.append(item)
        save_json(db, DB_FILE)
        return f"✅ Added {len(new_items)} items!", get_inventory_df()
    except Exception as e:
        return f"❌ Error: {str(e)}", get_inventory_df()

def chat_agent(message, history):
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    db = load_json(DB_FILE)

    # UPDATED: AI-Powered BULK Deletion Logic
    keywords = ["remove", "delete", "ate", "finished", "used", "clear"]
    if any(k in message.lower() for k in keywords):
        # We ask the AI to return a JSON list of items to be removed
        deletion_prompt = f"""
        User Message: '{message}'
        Current Pantry: {json.dumps(db)}

        Identify which items the user wants to remove.
        Return ONLY a JSON list of item names. If none match, return [].
        Example Output: ["Milk", "Eggs", "Butter"]
        """

        check = llm.invoke(deletion_prompt)
        try:
            # Clean the response to get the list
            json_str = check.content.replace('```json', '').replace('```', '').strip()
            items_to_remove = json.loads(json_str)

            if items_to_remove:
                used_log = load_json(USED_LOG)
                # Filter out the items to be deleted
                new_db = [i for i in db if i['Item'] not in items_to_remove]

                # Track what was removed for the analytics tab
                for name in items_to_remove:
                    used_log.append({"Item": name, "Date Used": datetime.now().strftime("%Y-%m-%d")})

                save_json(new_db, DB_FILE)
                save_json(used_log, USED_LOG)

                response_text = f"✅ I've removed the following from your inventory: {', '.join(items_to_remove)}."
                history.append((message, response_text))
                return "", history
        except:
            pass # If JSON fails, fall back to normal chat logic

    # Normal Recipe/Chat logic
    sys_msg = SystemMessage(content=f"Chef Mode. Pantry: {json.dumps(db)}. Suggest recipes or waste-reduction tips.")
    msgs = [sys_msg] + [HumanMessage(content=q) if i%2==0 else SystemMessage(content=a) for i, (q,a) in enumerate(history)] + [HumanMessage(content=message)]
    response = llm.invoke(msgs)
    history.append((message, response.content))
    return "", history

def manual_remove(item_name):
    db = load_json(DB_FILE)
    used_log = load_json(USED_LOG)
    for item in db:
        if item['Item'] == item_name:
            used_log.append({"Item": item_name, "Date Used": datetime.now().strftime("%Y-%m-%d")})
            break
    save_json([i for i in db if i['Item'] != item_name], DB_FILE)
    save_json(used_log, USED_LOG)
    return f"Removed {item_name}.", get_inventory_df()

def get_analytics():
    pantry = load_json(DB_FILE)
    used = load_json(USED_LOG)
    if not pantry and not used:
        return "Pantry is empty.", pd.DataFrame({"Status": ["No Data"], "Count": [0]})
    df_p = get_inventory_df()
    status_counts = df_p['Status'].value_counts().to_dict()
    final_data = {
        "Status": ["✅ Fresh", "🛑 Expired", "⚠️ At Risk", "🍴 Used"],
        "Count": [status_counts.get("✅ Fresh", 0), status_counts.get("🛑 Expired", 0), status_counts.get("⚠️ At Risk", 0), len(used)]
    }
    return "Pantry Health Updated.", pd.DataFrame(final_data)

In [ ]:
# --- 4. INTERFACE ---
with gr.Blocks(theme=gr.themes.Origin(primary_hue="emerald")) as demo:
    gr.Markdown("# 🍎 Byte-Zero: Pantry Tracker")

    with gr.Tabs():
        with gr.Tab("📖 Instructions"):
            gr.Markdown("## 📖 User Guide: Getting Started with Your Pantry Agent")
            gr.Markdown("""
            Welcome! This pilot program is designed to turn your pantry into a smart, waste-free environment.

            ### 📥 Tab 1: Inventory Management
            *Your digital record of everything in your kitchen.*
            - **Adding Items:** Use **Text Entry** to type grocery names or **Photo Entry** to upload an image of your items. Click **Update Pantry** to save.
            - **Inventory Table:** View everything you have added along with the date it was logged.
            - **Removing Items:** When you finish a food item, type the name and click **Manual Delete** or tell the Chat Agent.

            ### 📊 Tab 2: Food Waste Analytics
            *Insights into your habits.*
            - **What it does:** This section tracks your efficiency by comparing what you buy versus what you consume.
            -  Visual charts now track Fresh, Expired, and Used food.

            ### 💬 Tab 3: Chat with Agent
            *Your autonomous kitchen partner.*
            - **How to Chat:** Ask for recipes or say 'I drank the milk' to update your inventory.

            ### ⚠️SAFETY DISCLAIMER:
            Byte-Zero uses AI to estimate shelf life based on standard grocery averages.
            These dates are estimates only and do not account for storage conditions (e.g., temperature, humidity) or specific brand variations.
            Always verify freshness using the 'sell-by' date on the packaging and your own senses (sight/smell).
            """)

        with gr.Tab("📋 Inventory"):
            with gr.Row():
                with gr.Column():
                    t_in = gr.Textbox(label="Text Entry", placeholder="e.g. eggs, milk, bread")
                    img_in = gr.Image(label="Photo/Receipt", type="pil")
                    inv_btn = gr.Button("Update Pantry", variant="primary")
                    gr.Markdown("---")
                    delete_name = gr.Textbox(label="Remove Item (Manual)")
                    del_btn = gr.Button("Delete from Memory", variant="stop")
                    refresh_table = gr.Button("🔄 Refresh Table")
                with gr.Column():
                    inv_table = gr.Dataframe(value=get_inventory_df())
                    inv_status = gr.Textbox(label="Agent Status")

        with gr.Tab("📊 Analytics"):
            ana_btn = gr.Button("Update Analytics Charts")
            ana_summary = gr.Textbox(label="Summary")
            ana_plot = gr.BarPlot(x="Status", y="Count", color="Status", title="Pantry Lifecycle")

        with gr.Tab("💬 Chef Chat"):
            chatbot = gr.Chatbot(height=400)
            msg_in = gr.Textbox(label="Ask for recipes or tell the agent what you ate...")

  # BINDINGS
    inv_btn.click(inventory_agent, [t_in, img_in], [inv_status, inv_table])
    del_btn.click(manual_remove, [delete_name], [inv_status, inv_table])
    refresh_table.click(get_inventory_df, None, inv_table)
    ana_btn.click(get_analytics, None, [ana_summary, ana_plot])
    msg_in.submit(chat_agent, [msg_in, chatbot], [msg_in, chatbot])

demo.launch(debug=True)

/tmp/ipykernel_3386/3823124482.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Origin(primary_hue="emerald")) as demo:
/tmp/ipykernel_3386/3823124482.py:52: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_3386/3823124482.py:52: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d9b510690238261a26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
